# The shop that eats double is worth 2,400 a day. The other one is worth 420.

Two of the eight shops consume a single product, and the engine feeds a one-item shop at **twice** the rate of a multi-item one. That makes those towns an exclusive market for one good.

I found this because one of my routed schedules beats every strong opponent I have, 100% of the time, in exactly one kind of town. So I tried the same trick on the other single-item shop. It returned nothing. Then I tried it on a third product with a similar niche. Also nothing.

The rule that came out is not "route to the exclusive shop." It is that **double drain on a cheap product is still a cheap sink**, and one line of arithmetic separates the two cases before you spend anything.

## The consumption rule

Every shop consumes its list once every four turns. A shop with exactly one item consumes at double weight. With 24 turns in a day that is six pulls per day, doubled to twelve for the singles.

| shop | consumes | base price of each |
|---|---|---|
| **YARN_STORE** | WOOL | 200 |
| **PET_CAFE** | CARROT | 35 |
| BAKERY | EGG, WHEAT | 50, 25 |
| BRUNCH_SPOT | EGG, WHEAT, STRAWBERRY | 50, 25, 120 |
| FARMERS_MARKET | WHEAT, CARROT, TOMATO, STRAWBERRY | 25, 35, 60, 120 |
| ICE_CREAM_SHOP | STRAWBERRY, MILK, WHEAT | 120, 160, 25 |
| PIZZA_SHOP | MILK, TOMATO, WHEAT | 160, 60, 25 |
| SMOOTHIE_SHOP | STRAWBERRY, MILK | 120, 160 |

Both singles get the double. Only one of them is doubling something expensive.

## What it is worth in yarn towns

Almost every recorded schedule I have is a dairy economy: roughly 8 cows and 6 sheep, about 360 milk sold over a season. One of them is not. It runs **11 sheep and 6 cows and sells 315 wool**.

Routed into towns whose first shop is a yarn store, against the three strongest opponents I could find, rated 1959 to 2043:

| opponent rating | my record in yarn towns | median margin |
|---|---|---|
| 2043 | **14/14** | +15,534 |
| 2020 | **6/6** | +12,667 |
| 1959 | **6/6** | +12,714 |

In the same matches every other kind of town ran between 0% and 30%. Yarn towns are the only place I beat that class of opponent at all.

Two reasons, and they compound. Wool is the second most expensive good in the game. And because the field is dairy, almost nobody is selling wool into that sink, so the price stays up while everyone's milk craters.

## Rule out the boring explanation first

Is that schedule simply stronger? No. Give it **all eight** shop groups so it plays every town:

| configuration | ten band opponents, 200 games |
|---|---|
| wool schedule in yarn towns only | **137/200** |
| wool schedule everywhere | **56/200** |

The value is the match between schedule and town, not the schedule. Worth checking before building on any specialist: a 100% slice looks identical to a strong program until you run it outside the slice.

## The other single-item shop returns nothing

PET_CAFE eats CARROT, exclusively, at double rate. The schedule I route there plants **31 carrots** a season. The library has real carrot specialists: 159 to 247 planted, 417 to 723 sold.

Against the 2043-rated opponent, in pet-cafe towns:

| schedule | record | mean margin |
|---|---|---|
| what I already route there (31 carrots) | 3/10 | **-1,387** |
| carrot specialist, 191 planted | 4/10 | **-19,622** |
| carrot specialist, 247 planted | 0/10 | **-58,390** |

One extra win in ten and the money collapses.

The last row is a separate trap worth naming: that schedule comes from a lineage whose field layout does not match mine at the switch turn, so it inherits a farm it did not plant. Compare the two layouts at the switch day before you blame the idea.

## And a third product with the same shape

EGG has no dedicated shop, but two shops eat it and almost nobody makes it: most schedules keep three geese or none. There is a specialist in the library with **51 geese**.

| group | what I route there now | goose specialist |
|---|---|---|
| BAKERY, 10 towns | 1/10, -5,660 | 1/10, **-17,604** |
| BRUNCH_SPOT, 10 towns | 3/10, +1,458 | 3/10, **-9,992** |

Not one extra win, and the money halves or worse in both.

## The arithmetic that separates them

Drain per day times base price. Nothing else is needed.

YARN_STORE doubles a 200-coin good and drains **2,400 coins a day**, more than any other shop in the game. PET_CAFE doubles a 35-coin good and drains **420**, the *shallowest* of all eight despite the double. BAKERY, which eats two items at single rate, is deeper than PET_CAFE.

So "single-item shop" was never the right signal. It was the good's price all along, and the double is a multiplier on top of it. Run the cell and the ordering is unambiguous.

In [ ]:
# base prices and shop lists, taken from the engine's own constants
BASE_PRICE = {"WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120, "MELON": 250,
              "EGG": 50, "MILK": 160, "WOOL": 200, "FERTILIZER": 100}

SHOPS = {
    "YARN_STORE": ["WOOL"],
    "PET_CAFE": ["CARROT"],
    "BAKERY": ["EGG", "WHEAT"],
    "BRUNCH_SPOT": ["EGG", "WHEAT", "STRAWBERRY"],
    "FARMERS_MARKET": ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY"],
    "ICE_CREAM_SHOP": ["STRAWBERRY", "MILK", "WHEAT"],
    "PIZZA_SHOP": ["MILK", "TOMATO", "WHEAT"],
    "SMOOTHIE_SHOP": ["STRAWBERRY", "MILK"],
}

TURNS_PER_DAY, SHOP_PERIOD = 24, 4


def drain_per_day(shop):
    """Units and coins a shop removes from the market each day.

    A shop consumes its whole list every SHOP_PERIOD turns, and a shop with exactly
    one item consumes at double weight. The coin column is the one that ranks towns.
    """
    items = SHOPS[shop]
    weight = 2 if len(items) == 1 else 1
    pulls = TURNS_PER_DAY / SHOP_PERIOD
    units = weight * pulls * len(items)
    coins = sum(weight * pulls * BASE_PRICE[i] for i in items)
    return units, coins


rows = sorted(((s,) + drain_per_day(s) for s in SHOPS), key=lambda r: -r[2])
print(f"{'shop':16}{'items':>7}{'units/day':>12}{'coins/day':>12}")
for s, u, c in rows:
    single = " (double)" if len(SHOPS[s]) == 1 else ""
    print(f"{s:16}{len(SHOPS[s]):>7}{u:>12,.0f}{c:>12,.0f}{single}")

top, bottom = rows[0], rows[-1]
print()
print(f"deepest sink : {top[0]} at {top[2]:,.0f} coins/day")
print(f"shallowest   : {bottom[0]} at {bottom[2]:,.0f} coins/day -- and it is a DOUBLE shop")
print(f"ratio        : {top[2] / bottom[2]:.1f}x")

## The short version

A one-item shop drains its product at double rate, and that is real. But the double multiplies a price, so it is only worth chasing where the price is high.

Wool: 2,400 coins a day, thin supply because the field is dairy, and a specialist that goes 26/26 against opponents I otherwise lose to.

Carrot: 420 coins a day. Two specialists, one extra win in ten, money down by 19k and 58k.

Eggs: same story, zero extra wins.

If you have a specialist that dominates one slice, run it across every town before you believe it. Mine fell from 137/200 to 56/200 the moment it left the towns it fits, and that number is what told me what I actually had.